# Provisional benchmark on published warehouse data

Run **Runtime → Run all** in Google Colab. This notebook downloads a small published Access database, verifies its checksum, adapts its inventory/production/demand tables, and compares constrained allocation with deterministic greedy baselines.

**Important source limitation:** the source's `D` table columns are semantically mislabeled. This adapter uses a documented interpretation whose SKU joins were validated and whose raw values were independently checked with MDB export. The interpretation has **not been confirmed by the authors**. Results are provisional; inspect the displayed mapping and provenance before using them.

The documented demand interpretation uses `D.T` as day, `D.Q` as SKU, and `D.item_WEEK` as quantity. The shifted descriptive fields are `D.item_SKU` for batch, `D.item_BATCH` for month, and `D.item_MONTH` for week. These mappings are stated explicitly rather than relying on the misleading column names.

Dataset attribution: **Sirri, Accorsi, Lupi, and Manzini (2026)**, [DOI: 10.5281/zenodo.18229759](https://doi.org/10.5281/zenodo.18229759), [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/). The notebook adapts these data for a different allocation experiment; it does not reproduce the original paper's storage-optimization study.

All planning is local. No API keys, ERP actions, supplier messages, or external dispatch are involved. The initial download requires internet; a supplied local cache allows offline execution.

In [ ]:
import importlib
import importlib.util
import subprocess
import sys

needs_install = importlib.util.find_spec("assumption_ops") is None
if not needs_install:
    import assumption_ops
    needs_install = not hasattr(assumption_ops, "warehouse_scenario")
if needs_install or importlib.util.find_spec("access_parser") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--upgrade",
        "assumption-ops[datasets] @ git+https://github.com/crypticsymmetry/improved-giggle.git",
    ])
    for module_name in list(sys.modules):
        if module_name == "assumption_ops" or module_name.startswith("assumption_ops."):
            del sys.modules[module_name]
    importlib.invalidate_caches()

import json
import os
import platform
import shutil
from pathlib import Path
from pprint import pprint
from time import perf_counter

import scipy
from assumption_ops.baselines import compare_allocators
from assumption_ops.warehouse_data import download_warehouse, read_warehouse, warehouse_scenario


## 1. Obtain and verify the source data

For an offline local run, set `ASSUMPTION_OPS_WAREHOUSE_PATH` to your already downloaded `warehouse.accdb`. The default Colab run downloads the published database into the current runtime. Reading verifies the expected file checksum; an arbitrary replacement Access file is rejected.

`allow_inferred_demand=True` is an explicit opt-in to the provisional demand interpretation. It is required rather than silently assuming ambiguous column meanings.

In [ ]:
output_directory = Path("warehouse_benchmark_outputs")
output_directory.mkdir(exist_ok=True)
raw_path = output_directory / "warehouse.accdb"
local_cache = os.environ.get("ASSUMPTION_OPS_WAREHOUSE_PATH")
if local_cache:
    cached_path = Path(local_cache).expanduser().resolve()
    if cached_path != raw_path.resolve():
        shutil.copyfile(cached_path, raw_path)
elif not raw_path.exists():
    download_warehouse(raw_path)

data = read_warehouse(raw_path, allow_inferred_demand=True)
pprint(data.provenance)
pprint({
    "inventory_skus": len(data.initial_inventory),
    "production_records": len(data.production),
    "demand_records": len(data.demand),
    "production_days": len({record.day for record in data.production}),
    "demand_days": len({record.day for record in data.demand}),
})
assert data.provenance["inferred_demand_mapping"] is True
assert data.provenance["author_confirmation"] is False


## 2. Build the full observed-horizon allocation problem

Use all available SKUs and the full published 130-day horizon. Initial inventory and dated production provide supply; interpreted demand rows provide requested quantities and due days. Quantities come from the source, subject to the mapping caveat above.

This is an **offline, perfect-information snapshot**. Future production and demand are already known to the planner. It is not an online forecasting experiment or a causal operational evaluation.

Warehouse lanes, batching geometry, holding costs, and acquisition costs are not modeled from observed data. Unit acquisition cost is set to zero; configured penalty weights are illustrative preferences, not monetary values.

In [ ]:
native_scenario = warehouse_scenario(data, stock_fraction=1.0)
stress_scenario = warehouse_scenario(data, stock_fraction=0.5)
scenarios = {
    "published_quantities_provisional_mapping": native_scenario,
    "simulated_half_modeled_supply": stress_scenario,
}
for name, scenario in scenarios.items():
    pprint({
        "scenario": name,
        "supplies": len(scenario.supplies),
        "orders": len(scenario.orders),
        "requested_units": sum(order.quantity for order in scenario.orders),
        "supply_units": sum(supply.quantity for supply in scenario.supplies),
        "unit_acquisition_costs": sorted({supply.unit_cost for supply in scenario.supplies}),
        "policy": scenario.policy,
    })
    assert all(supply.unit_cost == 0 for supply in scenario.supplies)
assert sum(order.quantity for order in native_scenario.orders) == sum(
    order.quantity for order in stress_scenario.orders
)


## 3. Compare the actual allocators

The native case uses published inventory quantities. The second case scales every modeled supply quantity (initial inventory and production) to 50% as an explicitly **simulated stock stress**; it is not an observed stock-loss event. Both cases retain the published production and interpreted demand rows.

Each method receives identical inputs and is independently checked for feasibility and objective accounting. The methods are integer MILP, earliest-due-date allocation, and priority/cost greedy allocation.

In [ ]:
started = perf_counter()
reports = {
    name: compare_allocators(scenario.supplies, scenario.orders, scenario.policy)
    for name, scenario in scenarios.items()
}
environment = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "scipy": scipy.__version__,
    "total_comparison_wall_seconds": perf_counter() - started,
    "external_dispatch": False,
    "source_interpretation": "provisional, not author-confirmed",
}
pprint(environment)
for report in reports.values():
    assert report["external_dispatch"] is False


## 4. Inspect service, weighted score, and timing together

Weighted cost is evaluated under one policy per scenario. Fill rate uses allocated/requested units; late rate uses late/requested units. These are snapshot quantities, not repeated-horizon totals or measured business savings.

Method timing depends on the runtime and solver warm-up. A smaller weighted objective does not by itself establish commercial value.

In [ ]:
comparison_rows = []
for name, report in reports.items():
    for result in report["results"]:
        evaluation = result["evaluation"]
        metrics = evaluation["metrics"]
        denominator = max(1, metrics["requested_units"])
        comparison_rows.append({
            "scenario": name,
            "method": result["method"],
            "status": result["status"],
            "weighted_score": evaluation["costs"]["total"],
            "fill_rate": metrics["allocated_units"] / denominator,
            "late_rate": metrics["late_units"] / denominator,
            "unfilled_units": metrics["unfilled_units"],
            "on_time_units": metrics["on_time_units"],
            "gap_to_lp_bound": result["gap_to_lp_bound"],
            "elapsed_seconds": result["elapsed_seconds"],
        })
from IPython.display import Markdown, display

headers = ["Scenario", "Method", "Weighted score", "Fill", "Late", "LP gap", "Seconds"]
lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
for row in comparison_rows:
    gap = row["gap_to_lp_bound"]
    values = [
        row["scenario"], row["method"], f"{row['weighted_score']:,}",
        f"{row['fill_rate']:.2%}", f"{row['late_rate']:.2%}",
        "unavailable" if gap is None else f"{gap:,.0f}",
        f"{row['elapsed_seconds']:.4f}",
    ]
    lines.append("| " + " | ".join(values) + " |")
display(Markdown("\n".join(lines)))


## 5. Interpret the LP relaxation correctly

The LP is reported only as a lower bound, with fractional-variable diagnostics. It is not presented as an integer executable allocation plan.

This adapter creates a transportation-style allocation problem. With integral capacities and demand, its LP can already have integral optima. A tie between LP and MILP is expected in that regime and does not demonstrate an advantage from mixed-integer optimization. Stronger business constraints would change the problem and need their own evaluation.

In [ ]:
for name, report in reports.items():
    print(name)
    pprint(report["lp_relaxation"])
    assert "plan" not in report["lp_relaxation"]
    print("MILP proven optimal:", report["milp_proven_optimal"])
    pprint(report["limitations"])


## 6. Export inspectable results

JSON retains the allocator plans and evaluation details; CSV provides the comparison rows above. Both exports include or accompany the source provenance and methodological limitations. These are local artifacts, not external operational actions.

In [ ]:
import csv

report_path = output_directory / "warehouse_comparison.json"
report_path.write_text(
    json.dumps({
        "provenance": data.provenance,
        "environment": environment,
        "scenarios": reports,
        "stock_stress_note": "50% of all modeled supply is a simulated capacity stress, not an observed stock loss.",
        "method_note": "Offline perfect-information allocation; original storage study not reproduced.",
    }, indent=2) + "\n",
    encoding="utf-8",
)
csv_path = output_directory / "warehouse_comparison.csv"
with csv_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(comparison_rows[0]))
    writer.writeheader()
    writer.writerows(comparison_rows)
print(report_path)
print(csv_path)


## 7. Inspect a smaller SKU subset or change the simulated stress

The primary comparison above uses every SKU. This optional example selects the first twenty sorted inventory SKU identifiers and reuses the same verified source. Selecting a subset changes the workload; it does not create new observed data.

Before considering production use, confirm source semantics with the authors and model the actual operating constraints and costs. This provisional benchmark does not validate a deployment.

In [ ]:
subset_ids = tuple(sorted(data.initial_inventory)[:20])
subset_scenario = warehouse_scenario(data, sku_ids=subset_ids, stock_fraction=1.0)
pprint({
    "subset_skus": len(subset_ids),
    "subset_supplies": len(subset_scenario.supplies),
    "subset_orders": len(subset_scenario.orders),
})
# Optional additional run:
# subset_report = compare_allocators(
#     subset_scenario.supplies, subset_scenario.orders, subset_scenario.policy
# )
